# MRCD Pipeline - Full Workflow & Evaluation

Chạy full pipeline MRCD và đánh giá hiệu suất.

**Flow:**
1. Setup environment & load models
2. SLM Baseline evaluation
3. MRCD multi-round pipeline
4. Compare SLM vs MRCD

## 1. Setup

In [ ]:
%pip install ddgs rank_bm25 wikipedia transformers accelerate sentencepiece beautifulsoup4 curl_cffi sentence-transformers python-dotenv

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import set_seed
set_seed(42)

from src.config import *
print(f'LLM: {LLM_MODEL_NAME}')
print(f'SLM Backend: {SLM_BACKEND}')
print(f'Knowledge Mode: {KNOWLEDGE_MODE}')
print(f'Rounds: {NUM_LOOP}, Threshold: {CONFIDENCE_THRESHOLD}')

In [ ]:
# HuggingFace login (for gated models)
from huggingface_hub import login
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('HF logged in')
else:
    print('No HF_TOKEN found, skipping login')

## 2. Load Models & Data

In [ ]:
# Load LLM
from src.llm import get_llm
llm = get_llm()

# Load SLM
from src.slm import IntegratedSLM
slm = IntegratedSLM()

# Load data
from src.slm.dataset import load_data_from_csv
train_texts, train_labels, test_texts, test_labels = load_data_from_csv()

print(f'\nTrain: {len(train_texts)} samples')
print(f'Test:  {len(test_texts)} samples')

## 3. SLM Baseline Evaluation

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from src.utils import preprocess_text
from src.evaluation import evaluate_and_plot

if len(test_texts) > 0:
    print('Evaluating SLM baseline on test set...')
    slm_preds = []
    slm_confs = []

    for text in tqdm(test_texts, desc='SLM inference'):
        clean_text = preprocess_text(text)
        pred, conf, _ = slm.inference(clean_text)
        slm_preds.append(pred)
        slm_confs.append(conf)

    labels = ['Real/Truth', 'Fake/Rumor']
    metrics_slm = evaluate_and_plot(test_labels, slm_preds, labels=labels, model_name='SLM Baseline')

    print(f'\nAvg confidence: {np.mean(slm_confs):.4f}')
    print(f'Min: {np.min(slm_confs):.4f}, Max: {np.max(slm_confs):.4f}')
else:
    print('No test data available')

## 4. MRCD Pipeline

In [ ]:
from src.pipeline import run_mrcd_pipeline

if len(test_texts) > 0:
    flow_output = run_mrcd_pipeline(
        events=test_texts,
        slm=slm,
        reuse_knowledge_cache=True,
        # knowledge_mode='wiki_only',  # or 'full'
    )

    results = flow_output['results']
    history = flow_output['history']
    finalized_noisy = flow_output.get('finalized_noisy', [])

    print('\n=== Round History ===')
    for h in history:
        print(h)

    print(f'\nFinalized noisy by SLM: {len(finalized_noisy)}')
    print(f'Knowledge cache entries: {flow_output.get("knowledge_cache_size", 0)}')
else:
    print('No test data available')

## 5. MRCD Evaluation

In [ ]:
if len(test_texts) > 0:
    y_true = test_labels[:len(results)]
    y_pred = [r['label'] for r in results]

    labels = ['Real', 'Fake']
    metrics_mrcd = evaluate_and_plot(y_true, y_pred, labels=labels, model_name='MRCD Pipeline')

    # Sample predictions
    print('\n=== Sample Predictions ===')
    for i, r in enumerate(results[:10]):
        truth = 'Fake' if test_labels[i] == 1 else 'Real'
        pred = 'Fake' if r['label'] == 1 else 'Real'
        print(f'[id={r["event_id"]}] [{truth} -> {pred}] {r["text"][:60]}...')

## 6. Model Comparison

In [ ]:
from src.evaluation import compare_models

if 'metrics_slm' in dir() and 'metrics_mrcd' in dir():
    comparison_df = compare_models({
        'SLM Baseline': metrics_slm,
        'MRCD Pipeline': metrics_mrcd,
    })
else:
    print('Run baseline and MRCD evaluation first.')